# 07 · Deep learning: a small CNN on log-mel spectrograms

**Question:** does a CNN that learns its own features from the spectrogram beat the hand-made
features of notebooks 03-05 under exactly the same protocol (same 25 grouped splits, same metrics)?

**Input:** 64 log-mel bands over 0-4 kHz, 25 ms frames every 10 ms, a fixed 7 s per clip (crop or pad).
**Model:** 4 convolution blocks, pooling over time only (so pitch stays visible), dropout, one linear
layer; about 100k weights (`src/cnn.py`).
**Training,** inside each outer training part: AdamW, early stopping on a grouped validation part
carved out of the training part (the outer test fold is never used to pick the epoch), augmentation on
training batches only (time shift, low-level noise, SpecAugment). Two imbalance treatments:
square-root class-weighted loss, and a class-balanced sampler. A pitch-shift ablation (a shift along the
mel axis) runs on one repeat of the better treatment, because pitch may itself be a cue.

### How to run it on Google Colab
1. Here, once: `python scripts/make_colab_bundle.py` writes `data/colab/babycry_8k.zip` (the 479 clean
   clips, 8 kHz mono). Upload it to **My Drive/babycry/** in Google Drive.
2. Open this notebook in Colab (File → Open notebook → GitHub), choose **Runtime → Change runtime type → T4 GPU**,
   then **Runtime → Run all**. Allow Drive access when asked.
3. If the repository is private, add a read-only GitHub token as the Colab secret `GITHUB_TOKEN`
   (key icon in the left bar). It is read from the Secrets panel and never printed or saved in the notebook.
4. Results are saved to Drive after every split. If the session disconnects, run all cells again:
   finished splits are skipped.
5. Download from **My Drive/babycry/results/**: `predictions/cnn_*.csv` and `cnn_training_log.csv`, plus
   the executed notebook (File → Download → .ipynb). Put them in the repository's `results/` and
   `notebooks/` folders.

The notebook also runs locally (CPU, about 1-2 minutes per split): it then reads the audio through
`.env` like the other notebooks and keeps its per-split files in `data/cnn_runs/` (gitignored).

## 1. Setup

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/techatanu/Infant-State-Recognition-System.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/babycry")
    REPO = Path("/content/Infant-State-Recognition-System")
    if not REPO.exists():
        url = REPO_URL
        try:                                    # only needed for a private repository
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        done = subprocess.run(["git", "clone", "--quiet", url, str(REPO)], capture_output=True)
        if done.returncode != 0:                # never echo the URL: it may hold the token
            raise RuntimeError("git clone failed; for a private repo set the GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-dl.txt")], check=True)
    DATA = Path("/content/data")
    if not DATA.exists():                       # unzip on the VM's disk: reading many small files from Drive is slow
        shutil.copy(DRIVE / "babycry_8k.zip", "/content/babycry_8k.zip")
        shutil.unpack_archive("/content/babycry_8k.zip", DATA)
    os.environ["BABYCRY_DATA_DIR"] = str(DATA)
    RUNS = DRIVE / "results" / "cnn_runs"
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUNS = REPO / "data" / "cnn_runs"
RUNS.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"python {sys.version.split()[0]}, torch {torch.__version__}, device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else f" ({torch.get_num_threads()} CPU threads)"))
print(f"repo commit {commit}, per-split results in {RUNS}")

The CNN code is tested here with the same tests as on the development machine (they check, for
example, that the torch spectrogram equals the numpy reference and that the network learns pure tones).

In [ ]:
import importlib.util

if importlib.util.find_spec("pytest") is None:
    print("pytest is not installed in this environment; skipping the CNN tests")
else:
    tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_cnn.py"], capture_output=True, text=True)
    print(tests.stdout.strip().splitlines()[-1] if tests.returncode == 0 else tests.stdout[-3000:])
    assert tests.returncode == 0, "CNN tests failed: see above"

In [ ]:
import warnings

from src.audio import CLASSES, load_audio, load_clean_index
from src.cnn import N_SAMPLES, VARIANTS, fix_length, prediction_table, train_split
from src.metrics import (balanced_accuracy, cluster_bootstrap, encode, interval, macro_f1, paired_bootstrap,
                         per_class_table, predictions_by_repeat, repeat_scores)
from src.splits import PREDICTIONS_DIR, load_folds, load_predictions, outer_splits, repeats, save_predictions

warnings.filterwarnings("ignore", message="The least populated class")   # 2-clip classes in the validation split
pd.set_option("display.width", 160)
idx = load_clean_index()
folds = load_folds(idx)
t, groups = encode(idx["label"]), idx["source"].to_numpy()
start = time.time()
waves = np.stack([fix_length(load_audio(p)) for p in idx["path"]])
print(f"{len(idx)} clips loaded as {waves.shape} float32 in {time.time() - start:.0f} s "
      f"({(idx['duration_s'] < 7).sum()} padded, {(idx['duration_s'] > 7).sum()} cropped to 7 s)")

## 2. Train on every split

Settings (from `src/cnn.py`, fixed before any run): batch 32, AdamW with learning rate 1e-3 and weight decay
1e-2, at most 80 epochs, stopping after 15 epochs without a better validation score. `REPEATS` can be cut to
`["r0"]` if the session budget is too small; the results then say so.

Each split's predictions and training curve are written to `RUNS` as soon as it finishes; splits already
there are skipped, so a disconnected session resumes where it stopped.

In [ ]:
REPEATS = repeats(folds)                  # all 5; ["r0"] for a shorter run
PITCH_ABLATION = True                     # one repeat of the better treatment with a mel-axis shift of up to 2 bands


def run(variant, repeat_list, tag=None, **overrides):
    tag = tag or variant
    (RUNS / tag).mkdir(exist_ok=True)
    for r in repeat_list:
        for k, train, test in outer_splits(folds, r):
            out = RUNS / tag / f"{r}_fold{k}.csv"
            if out.exists():
                continue
            seed = int(r[1:]) * 10 + k
            proba, log, history = train_split(waves, t, groups, train, test, variant, seed, DEVICE, **overrides)
            history.to_csv(RUNS / tag / f"{r}_fold{k}_history.csv", index=False)
            pd.DataFrame([{**log, "tag": tag, "repeat": r, "fold": k, "commit": commit, "device": DEVICE}]).to_csv(
                RUNS / tag / f"{r}_fold{k}_log.csv", index=False)
            prediction_table(proba, t, test, r, k).to_csv(out, index=False, float_format="%.6g")
            print(f"{tag:28s} {r} fold {k}: best epoch {log['best_epoch']:2d} of {log['epochs_run']:2d}, "
                  f"{log['seconds']:4.0f} s")


def collect(tag):
    files = sorted((RUNS / tag).glob("r*_fold?.csv"))
    pred = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    log = pd.concat([pd.read_csv(f) for f in sorted((RUNS / tag).glob("r*_fold?_log.csv"))], ignore_index=True)
    return pred, log


for variant in VARIANTS:
    run(variant, REPEATS)

In [ ]:
predictions, logs = {}, []
for variant in VARIANTS:
    predictions[variant], log = collect(variant)
    logs.append(log)
    assert len(predictions[variant]) == len(idx) * len(REPEATS), f"{variant}: missing splits"
    save_predictions(f"cnn_{variant}", predictions[variant], idx)
training_log = pd.concat(logs, ignore_index=True)
training_log.to_csv(REPO / "results" / "cnn_training_log.csv", index=False)
if IN_COLAB:                               # copies to download from Drive
    (DRIVE / "results" / "predictions").mkdir(parents=True, exist_ok=True)
    for variant in VARIANTS:
        shutil.copy(PREDICTIONS_DIR / f"cnn_{variant}.csv", DRIVE / "results" / "predictions")
    shutil.copy(REPO / "results" / "cnn_training_log.csv", DRIVE / "results")
print(training_log.groupby("variant")[["best_epoch", "epochs_run", "seconds"]].describe().round(1).T)
print(f"commit(s) {sorted(training_log['commit'].unique())}, device(s) {sorted(training_log['device'].unique())}")

## 3. Scores, on the same footing as notebook 05

Means over repeats (± std), 95% cluster-bootstrap intervals over babies, and the paired difference from the
always-hungry baseline. GPU training is not bit-for-bit reproducible, so the spread over repeats is the
honest measure of run-to-run variation.

In [ ]:
majority_pred = load_predictions("majority", idx).loc[lambda d: d["repeat"].isin(REPEATS)]
majority = predictions_by_repeat(majority_pred, len(idx))


def evaluate(pred, n_boot=1000):
    per_rep = repeat_scores(pred)
    preds = predictions_by_repeat(pred, len(idx))
    row = {m: per_rep[m].mean() for m in per_rep.columns}
    row.update({f"{m}_std": per_rep[m].std(ddof=1) for m in per_rep.columns})
    for m, fn in [("balanced_accuracy", balanced_accuracy), ("macro_f1", macro_f1)]:
        row[f"{m}_low"], row[f"{m}_high"] = interval(cluster_bootstrap(t, preds, groups, fn, n_boot, seed=0))
    d = paired_bootstrap(t, preds, majority, groups, n_boot=n_boot, seed=0)
    row["ba_vs_majority"], (row["ba_vs_majority_low"], row["ba_vs_majority_high"]) = d.mean(), interval(d)
    return row


rows = [{"model": "majority", "variant": "baseline", **evaluate(majority_pred)}]
rows += [{"model": "cnn", "variant": v, **evaluate(p)} for v, p in predictions.items()]
scores = pd.DataFrame(rows).set_index(["model", "variant"])
pd.DataFrame({
    "balanced acc": scores["balanced_accuracy"].map("{:.3f}".format) + " ± " + scores["balanced_accuracy_std"].map("{:.3f}".format),
    "BA 95% CI": scores["balanced_accuracy_low"].map("{:.3f}".format) + "-" + scores["balanced_accuracy_high"].map("{:.3f}".format),
    "macro-F1": scores["macro_f1"].map("{:.3f}".format),
    "BA vs majority (95% CI)": scores["ba_vs_majority"].map("{:+.3f}".format) + " (" + scores["ba_vs_majority_low"].map("{:+.3f}".format)
                               + ", " + scores["ba_vs_majority_high"].map("{:+.3f}".format) + ")",
    "accuracy": scores["accuracy"].round(3), "log-loss": scores["log_loss"].round(3), "top-3": scores["top3_accuracy"].round(3),
})

In [ ]:
best_variant = scores.loc["cnn"]["balanced_accuracy"].idxmax()
pred = predictions[best_variant]
tables = [per_class_table(encode(g.sort_values("clip")["y_true"]), encode(g.sort_values("clip")["y_pred"]), groups)
          for _, g in pred.groupby("repeat")]
per_class = pd.concat([tbl[["predicted", "precision", "recall", "f1"]] for tbl in tables]).groupby(level=0, sort=False).mean()
per_class.insert(0, "babies", tables[0]["babies"])
per_class.insert(0, "clips", tables[0]["clips"])
per_class["reliable (5+ babies)"] = tables[0]["reliable"]
print(f"per class, {best_variant} (mean over repeats):")
per_class.round(3)

## 4. Format check (as in notebook 06)

*cold_hot*, *lonely* and *scared* exist only as 44.1 kHz recordings. If the CNN uses the recording format, the
few 44.1 kHz clips of *tired*, *discomfort* and *burping* get more probability on those three classes than the
8 kHz clips of the same classes do. AUROC 0.5 = no format effect; the random forest in notebook 06 scored 0.81.

In [ ]:
from sklearn.metrics import roc_auc_score

three = ["p_cold_hot", "p_lonely", "p_scared"]
for variant, p in predictions.items():
    p3 = p.assign(p3=p[three].sum(axis=1)).groupby("clip")["p3"].mean()
    check = idx.join(p3).loc[lambda d: d["label"].isin(["tired", "discomfort", "burping"])]
    is44 = check["sample_rate"] == 44100
    print(f"{variant:18s} mean p(cold_hot+lonely+scared): 44.1 kHz {check.loc[is44, 'p3'].mean():.3f}, "
          f"8 kHz {check.loc[~is44, 'p3'].mean():.3f}; AUROC {roc_auc_score(is44, check['p3']):.2f} "
          f"({is44.sum()} vs {(~is44).sum()} clips)")

## 5. Pitch-shift ablation

The better treatment again on repeat r0 only, with a random shift of up to ±2 mel bands added to the
augmentation (roughly ±3-6% in frequency in the cry range). It is compared with the same 5 splits without the shift.

In [ ]:
if PITCH_ABLATION:
    tag = f"{best_variant}_pitch"
    run(best_variant, ["r0"], tag=tag, pitch_shift_bands=2)
    pitch, pitch_log = collect(tag)
    pitch_log.to_csv(REPO / "results" / "cnn_pitch_ablation_log.csv", index=False)
    save_predictions(f"cnn_{tag}", pitch, idx)
    if IN_COLAB:
        shutil.copy(PREDICTIONS_DIR / f"cnn_{tag}.csv", DRIVE / "results" / "predictions")
        shutil.copy(REPO / "results" / "cnn_pitch_ablation_log.csv", DRIVE / "results")
    same = predictions[best_variant].loc[lambda d: d["repeat"] == "r0"]
    order = lambda d: d.sort_values("clip")
    a, b = order(pitch), order(same)
    d = paired_bootstrap(t, encode(a["y_pred"]), encode(b["y_pred"]), groups, n_boot=1000, seed=0)
    print(f"repeat r0: BA without shift {balanced_accuracy(t, encode(b['y_pred'])):.3f}, "
          f"with shift {balanced_accuracy(t, encode(a['y_pred'])):.3f}, "
          f"difference {d.mean():+.3f} (95% CI {interval(d)[0]:+.3f}, {interval(d)[1]:+.3f})")

## 6. Findings

TODO: written from this notebook's outputs after the Colab run. No CNN result exists yet.